### Student Name: Feliciann Elliot
### Course: MAI5301 - Foundations Of Large Language Models
### Activity: Assigment #3

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List, Literal

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class LayerNorm(nn.Module):
    """
    LayerNorm implemented from scratch.
    """
    def __init__(self, dim: int, eps: float = 1e-5, elementwise_affine: bool = True):
        super().__init__()
        self.dim = dim
        self.eps = eps
        self.elementwise_affine = elementwise_affine

        if elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(dim))
            self.beta = nn.Parameter(torch.zeros(dim))
        else:
            self.register_parameter("gamma", None)
            self.register_parameter("beta", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, C) or (..., C)
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        if self.elementwise_affine:
            return x_hat * self.gamma + self.beta
        return x_hat


def sanity_check_layernorm(device: str = "cpu") -> None:
    torch.manual_seed(0)
    B, T, C = 2, 4, 8
    x = torch.randn(B, T, C, device=device)

    ln_s = LayerNorm(C).to(device)
    ln_t = nn.LayerNorm(C).to(device)

    # Make sure both use the same affine parameters for a fair comparison
    with torch.no_grad():
        ln_t.weight.copy_(ln_s.gamma)
        ln_t.bias.copy_(ln_s.beta)

    ys = ln_s(x)
    yt = ln_t(x)

    max_abs_diff = (ys - yt).abs().max().item()
    print(f"[LayerNorm check] max_abs_diff = {max_abs_diff:.8f} (should be ~0)")


In [ ]:
def gelu_approx(x: torch.Tensor) -> torch.Tensor:
    """
    Common GELU approximation used in GPT-style models:
    0.5*x*(1+tanh(sqrt(2/pi)*(x+0.044715*x^3)))
    """
    return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * x**3)))


class GELUScratch(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return gelu_approx(x)


def sanity_check_gelu(device: str = "cpu") -> None:
    torch.manual_seed(0)
    x = torch.randn(1024, device=device)

    y_ours = gelu_approx(x)
    y_torch = F.gelu(x)  # exact-ish implementation in PyTorch

    max_abs_diff = (y_ours - y_torch).abs().max().item()
    mean_abs_diff = (y_ours - y_torch).abs().mean().item()

    print(f"[GELU check] max_abs_diff={max_abs_diff:.8f}, mean_abs_diff={mean_abs_diff:.8f}")
    # Quick ReLU vs GELU vibe check (not a 'test', just observation)
    y_relu = F.relu(x)
    print(f"[ReLU vs GELU] mean(ReLU)={y_relu.mean().item():.6f}, mean(GELU)={y_torch.mean().item():.6f}")


In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int
    context_length: int
    d_model: int
    n_heads: int
    n_layers: int
    dropout: float = 0.1
    bias: bool = True  # allow bias in linear layers

    @staticmethod
    def small(vocab_size: int, context_length: int, dropout: float = 0.1) -> "GPTConfig":
        # Exercise 4.1: Small GPT-like config
        return GPTConfig(
            vocab_size=vocab_size,
            context_length=context_length,
            d_model=384,
            n_heads=6,
            n_layers=6,
            dropout=dropout,
            bias=True,
        )

    @staticmethod
    def medium(vocab_size: int, context_length: int, dropout: float = 0.1) -> "GPTConfig":
        # Exercise 4.1: Medium GPT-like config
        return GPTConfig(
            vocab_size=vocab_size,
            context_length=context_length,
            d_model=768,
            n_heads=12,
            n_layers=12,
            dropout=dropout,
            bias=True,
        )

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0, "d_model must be divisible by n_heads"
        self.cfg = cfg
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads

        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

        # causal mask as a buffer (not learnable)
        mask = torch.triu(torch.ones(cfg.context_length, cfg.context_length), diagonal=1).bool()
        self.register_buffer("causal_mask", mask, persistent=True)

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: Optional[Dict[str, torch.Tensor]] = None,
        use_kv_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]]]:
        """
        x: (B, T, C)
        kv_cache (optional): {"k": (B, H, T_cached, D), "v": (B, H, T_cached, D)}
        If use_kv_cache: append new k/v and return updated cache
        """
        B, T, C = x.shape

        qkv = self.qkv(x)  # (B, T, 3C)
        q, k, v = qkv.chunk(3, dim=-1)

        # reshape to heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, H, T, D)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, H, T, D)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, H, T, D)

        if use_kv_cache:
            if kv_cache is None:
                kv_cache = {"k": k, "v": v}
            else:
                # append along sequence dimension
                kv_cache["k"] = torch.cat([kv_cache["k"], k], dim=2)
                kv_cache["v"] = torch.cat([kv_cache["v"], v], dim=2)
            k_all = kv_cache["k"]
            v_all = kv_cache["v"]
            T_all = k_all.size(2)
        else:
            k_all, v_all = k, v
            T_all = T

        att = (q @ k_all.transpose(-2, -1)) / math.sqrt(self.head_dim)


        assert T_all <= self.cfg.context_length, "Exceeded context_length in KV-cache"

        row_start = T_all - T
        mask = self.causal_mask[row_start:row_start + T, :T_all]  # (T, T_all)
        att = att.masked_fill(mask.view(1, 1, T, T_all), float("-inf"))

        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v_all  # (B, H, T, D)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)

        y = self.resid_dropout(self.out(y))
        return y, kv_cache if use_kv_cache else None

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        hidden = 4 * cfg.d_model  # expansion ratio 4x
        self.fc1 = nn.Linear(cfg.d_model, hidden, bias=cfg.bias)
        self.act = GELUScratch()
        self.fc2 = nn.Linear(hidden, cfg.d_model, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = LayerNorm(cfg.d_model)
        self.attn = MultiHeadSelfAttention(cfg)
        self.ln2 = LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg)

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: Optional[Dict[str, torch.Tensor]] = None,
        use_kv_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]]]:
        attn_out, kv_cache = self.attn(self.ln1(x), kv_cache=kv_cache, use_kv_cache=use_kv_cache)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, kv_cache

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module) -> None:

        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        idx: torch.Tensor,
        kv_caches: Optional[List[Dict[str, torch.Tensor]]] = None,
        use_kv_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[List[Dict[str, torch.Tensor]]]]:

        B, T = idx.shape
        assert T <= self.cfg.context_length, "Sequence length exceeds context_length"

        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)  # (1, T)

        x = self.tok_emb(idx) + self.pos_emb(pos)  # (B, T, C)
        x = self.drop(x)

        new_caches = [] if use_kv_cache else None
        for layer_i, block in enumerate(self.blocks):
            layer_cache = None
            if use_kv_cache:
                if kv_caches is not None:
                    layer_cache = kv_caches[layer_i]
            x, updated_cache = block(x, kv_cache=layer_cache, use_kv_cache=use_kv_cache)
            if use_kv_cache:
                new_caches.append(updated_cache if updated_cache is not None else {"k": None, "v": None})

        x = self.ln_f(x)
        logits = self.head(x)
        return logits, new_caches

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
        mode: Literal["greedy", "sample"] = "greedy",
        use_kv_cache: bool = False,
    ) -> torch.Tensor:

        self.eval()
        kv_caches: Optional[List[Dict[str, torch.Tensor]]] = None

        for _ in range(max_new_tokens):
            # if no cache, feed full context window; if cache, feed only last token
            if use_kv_cache and idx.size(1) > 1:
                idx_cond = idx[:, -1:]  # incremental token
            else:
                idx_cond = idx[:, -self.cfg.context_length :]

            logits, kv_caches = self.forward(idx_cond, kv_caches=kv_caches, use_kv_cache=use_kv_cache)
            logits = logits[:, -1, :]  # (B, vocab)

            if temperature != 1.0:
                logits = logits / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, k=top_k, dim=-1)
                min_v = v[:, -1].unsqueeze(-1)
                logits = torch.where(logits < min_v, torch.full_like(logits, float("-inf")), logits)

            probs = F.softmax(logits, dim=-1)

            if mode == "greedy":
                next_token = torch.argmax(probs, dim=-1, keepdim=True)
            else:
                next_token = torch.multinomial(probs, num_samples=1)

            idx = torch.cat([idx, next_token], dim=1)

            # stop if context_length exceeded for absolute pos embeddings
            if idx.size(1) >= self.cfg.context_length and not use_kv_cache:
                # without cache we can still keep sliding; with absolute pos embeddings,
                # we must keep within context_length for correct positions.
                idx = idx[:, -self.cfg.context_length:]

        return idx

In [ ]:
# Exercise 4.4: Memory estimation

def estimate_memory_bytes(model: nn.Module, dtype: torch.dtype = torch.float32) -> int:

    bytes_per_param = torch.tensor([], dtype=dtype).element_size()
    total_params = sum(p.numel() for p in model.parameters())
    return total_params * bytes_per_param


def estimate_training_memory_bytes_adam(model: nn.Module, dtype: torch.dtype = torch.float32) -> int:

    base = estimate_memory_bytes(model, dtype=dtype)
    # params (1x) + grads (1x) + m (1x) + v (1x) = ~4x
    return 4 * base

In [ ]:
def main() -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    # Sanity checks: LayerNorm and GELU
    sanity_check_layernorm(device=device)
    sanity_check_gelu(device=device)

    # Minimal vocab/context just to show generation works
    vocab_size = 1000
    context_length = 64

    # Exercise 4.1: small vs medium
    cfg_small = GPTConfig.small(vocab_size=vocab_size, context_length=context_length, dropout=0.1)
    cfg_medium = GPTConfig.medium(vocab_size=vocab_size, context_length=context_length, dropout=0.1)

    model_small = GPTModel(cfg_small).to(device)
    model_medium = GPTModel(cfg_medium).to(device)

    print("\n[Model sizes]")
    print("small params:", sum(p.numel() for p in model_small.parameters()))
    print("medium params:", sum(p.numel() for p in model_medium.parameters()))

    print("\n[Memory estimates - params only]")
    print("small fp32 bytes:", estimate_memory_bytes(model_small, torch.float32))
    print("medium fp32 bytes:", estimate_memory_bytes(model_medium, torch.float32))

In [ ]:
if __name__ == "__main__":
    main()

Device: cpu
[LayerNorm check] max_abs_diff = 0.00000012 (should be ~0)
[GELU check] max_abs_diff=0.00047330, mean_abs_diff=0.00008743
[ReLU vs GELU] mean(ReLU)=0.420953, mean(GELU)=0.304491

[Model sizes]
small params: 11440128
medium params: 86641152

[Memory estimates - params only]
small fp32 bytes: 45760512
medium fp32 bytes: 346564608
